In [2]:
!pip install transformers datasets torch numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# 1단계: 필요한 라이브러리 임포트
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [7]:
# 2단계: 데이터셋 로드 및 전처리
print("데이터셋 로드 시작")
dataset = load_dataset("nyu-mll/glue", "mnli")
print("데이터셋 로드 완료")


데이터셋 로드 시작
데이터셋 로드 완료


In [21]:
train_val_split = dataset["train"].train_test_split(test_size=0.02, seed=42)  # 2%만 validation으로
train_dataset = train_val_split["train"].select(range(10000))  # 10,000개 샘플만 사용
val_dataset = train_val_split["test"]
print(f"학습 데이터 크기: {len(train_dataset)}")
print(f"검증 데이터 크기: {len(val_dataset)}")

학습 데이터 크기: 10000
검증 데이터 크기: 7855


In [22]:
# 3단계: 토크나이저 설정
print("토크나이저 설정 시작")
model_name = "bert-base-uncased"  # 더 작은 모델 사용
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# 데이터셋 토큰화
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
print("토큰화 완료")

토크나이저 설정 시작


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7855 [00:00<?, ? examples/s]

토큰화 완료


In [23]:
# 4단계: 모델 설정
print("모델 설정 시작")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)
print("모델 설정 완료")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


모델 설정 시작


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


모델 설정 완료


In [24]:
# 5단계: 평가 메트릭 함수 정의
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [25]:
# 6단계: 트레이너 설정
print("트레이너 설정 시작")
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=32,  # 배치 크기 더 증가
    per_device_eval_batch_size=32,
    num_train_epochs=2,  # 에포크 수 더 감소
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)
print("트레이너 설정 완료")

트레이너 설정 시작
트레이너 설정 완료


In [26]:
# 7단계: 학습 실행
print("학습 시작")
trainer.train()
print("학습 완료")


학습 시작


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.738113,0.685423
2,0.813400,0.698187,0.713431


학습 완료


In [27]:
# 8단계: validation_matched에 대한 평가
print("validation_matched 평가 시작")
# 전체 validation_matched 데이터셋 사용
tokenized_val_matched = dataset["validation_matched"].map(tokenize_function, batched=True)
eval_results = trainer.evaluate(tokenized_val_matched)
print(f"validation_matched 평가 결과: {eval_results}")
print(f"validation_matched 정확도: {eval_results['eval_accuracy']:.4f}")

validation_matched 평가 시작


Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

validation_matched 평가 결과: {'eval_loss': 0.6985036134719849, 'eval_accuracy': 0.7174732552215995, 'eval_runtime': 64.7554, 'eval_samples_per_second': 151.57, 'eval_steps_per_second': 4.741, 'epoch': 2.0}
validation_matched 정확도: 0.7175
